# A Reproducible Comparative Study of Physics-Informed Neural Networks, Adaptive Sampling, and Deep Operator Learning for ODE Systems

**A survey-style computational review with unified experiments drawn from the supplied PINN notebooks**  
**Literature snapshot:** 27 July 2026  

## Abstract

Physics-informed neural networks (PINNs) recast the solution of differential equations as an optimization problem in which automatic differentiation enforces the governing law. Their promise is especially attractive for inverse problems, sparse observations, and continuous surrogate representations; however, their training can be ill-conditioned, sensitive to loss balance, and inefficient when residual error is spatially or temporally localized. This notebook is a self-contained review and reproducible benchmark study of standard PINNs, residual-based adaptive refinement (RAR), residual-adaptive/importance sampling (RAS), DeepONet, and physics-informed DeepONet (PI-DeepONet) on first- and second-order ordinary differential equations and coupled ODE systems. It reimplements the core ideas in the supplied notebooks in one common PyTorch framework, standardizes nondimensionalization, initial-condition treatment, reference solutions, evaluation grids, random seeds, and reporting. The benchmark contains exponential decay, a damped oscillator, logistic growth, consecutive reaction kinetics, Lotka–Volterra dynamics, an SIR epidemic model, the Van der Pol oscillator, quasi-steady-state enzyme kinetics, and a Monod bioprocess model. It produces publication-ready solution, error, loss, residual, sampling, convergence, accuracy–efficiency, and summary-table outputs automatically. The study is deliberately honest about scope: it is an ODE-focused computational comparison, not a substitute for PDE-scale benchmarking or a claim of universal method superiority.

**Keywords:** scientific machine learning; physics-informed neural networks; PINNs; residual-based adaptive refinement; adaptive sampling; importance sampling; DeepONet; neural operators; ODE systems; inverse problems; reproducible benchmarking.

> **Pre-execution note.** The quantitative conclusions in this notebook are generated from the executed experiment cells below. The narrative deliberately avoids pre-populating unverified performance claims.

## 1. Introduction

Ordinary differential equations (ODEs) encode time evolution in kinetics, epidemiology, population dynamics, oscillations, and bioprocesses. Classical time integrators are usually the appropriate accuracy-and-cost baseline when the equation is known and one trajectory is required. Neural approaches become attractive when a differentiable surrogate, sparse noisy observations, unknown parameters, a solution family, or coupling to other learned components is required.

The modern PINN formulation represents a state trajectory by a neural network and penalizes violations of the governing equation at collocation points. Automatic differentiation avoids finite-difference differentiation of the neural surrogate. This approach is conceptually related to the hard-constraint trial functions introduced for ODE/PDE solution by Lagaris, Likas, and Fotiadis [L1], and was popularized in the continuous- and discrete-time PINN framework of Raissi, Perdikaris, and Karniadakis [L2].

The central practical difficulty is not expressing the residual; it is optimizing a multi-objective loss reliably. Stiffness, disparate state scales, oscillation, sharp transient layers, long time horizons, and a poor distribution of collocation points can leave important portions of a trajectory under-resolved. Gradient-flow and neural-tangent-kernel analyses explain part of this behavior [L5, L6]. Adaptive sampling responds by allocating collocation effort where the current model has large residuals. Operator learning changes the amortization question: instead of re-training for each parameter vector, it learns a map from a family of inputs (parameters, initial conditions, or forcing functions) to a family of trajectories [L18–L22].

This notebook therefore asks four bounded questions:

1. Under matched normalization and training budgets, how do a standard PINN, RAR, and residual-adaptive sampling behave on the supplied ODE families?
2. When an operator must answer many parameter queries, how do data-driven DeepONet and PI-DeepONet compare at a held-out target parameter?
3. Where do adaptive points concentrate, and does that concentration coincide with reduced residual/error?
4. Which conclusions are robust enough to report, and which remain configuration-dependent?

In [ ]:
# 0. Reproducible dependency bootstrap
# This cell is intentionally self-contained: on a fresh Jupyter kernel it installs
# the small set of packages used by the study, then continues without manual steps.
import importlib.util
import subprocess
import sys

REQUIRED = {
    "numpy": "numpy>=1.24",
    "pandas": "pandas>=2.0",
    "scipy": "scipy>=1.10",
    "matplotlib": "matplotlib>=3.7",
    "seaborn": "seaborn>=0.13",
    "torch": "torch>=2.1",
}

missing = [spec for module, spec in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing runtime dependencies:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("All required runtime dependencies are already available.")

In [ ]:
# 1. Imports, deterministic settings, visual style, and study configuration
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from time import perf_counter
import json
import math
import os
import platform
import random
import sys
import warnings

try:
    import resource
except ImportError:  # Windows lacks the POSIX resource module.
    resource = None

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from scipy.integrate import solve_ivp
from scipy.stats import qmc
import torch
import torch.nn as nn

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 20260727
RUN_PROFILE = os.environ.get("PINN_STUDY_PROFILE", "quick").lower()
# Valid profiles: "quick" (all methods/all benchmarks, CPU-friendly) and
# "paper" (longer Adam + L-BFGS training for final reported numbers).
assert RUN_PROFILE in {"quick", "paper"}, "PINN_STUDY_PROFILE must be 'quick' or 'paper'."

OUTPUT_DIR = Path("pinn_ode_study_outputs")
FIG_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
torch.set_default_dtype(DTYPE)

COLORS = {
    "PINN-LHS": "#2C6DB2",
    "RAR": "#C84A44",
    "RAS": "#7A5AA6",
    "DeepONet": "#2F8F6B",
    "PI-DeepONet": "#D4872F",
    "Exact / reference": "#222222",
}

sns.set_theme(style="whitegrid", context="notebook")
mpl.rcParams.update({
    "figure.dpi": 115,
    "savefig.dpi": 240,
    "font.family": "DejaVu Sans",
    "axes.titleweight": "semibold",
    "axes.labelcolor": "#20242A",
    "xtick.color": "#30343B",
    "ytick.color": "#30343B",
    "axes.edgecolor": "#9AA2AD",
    "grid.color": "#DEE2E7",
    "grid.linewidth": 0.7,
    "legend.frameon": True,
    "legend.framealpha": 0.94,
})

PROFILE = {
    "quick": dict(
        width=32, depth=3, n_collocation=64, pinn_epochs=260,
        rar_initial=24, rar_rounds=3, rar_epochs=100, rar_add=14,
        candidate_pool=480, ras_resample_every=35, ras_mix_uniform=0.20,
        operator_width=40, operator_latent=40, operator_train_params=48,
        operator_train_times=40, operator_batch_params=18, operator_batch_times=18,
        deeponet_epochs=260, pi_deeponet_epochs=260, pi_physics_weight=0.30,
        pi_collocation_params=10, pi_collocation_times=12, log_every=20,
        lbfgs_max_iter=0, eval_points=241,
    ),
    "paper": dict(
        width=64, depth=4, n_collocation=256, pinn_epochs=2200,
        rar_initial=48, rar_rounds=6, rar_epochs=420, rar_add=24,
        candidate_pool=2500, ras_resample_every=80, ras_mix_uniform=0.12,
        operator_width=80, operator_latent=80, operator_train_params=160,
        operator_train_times=96, operator_batch_params=40, operator_batch_times=36,
        deeponet_epochs=1800, pi_deeponet_epochs=1800, pi_physics_weight=0.35,
        pi_collocation_params=24, pi_collocation_times=24, log_every=50,
        lbfgs_max_iter=250, eval_points=401,
    ),
}[RUN_PROFILE]

ACTIVE_METHODS = ["PINN-LHS", "RAR", "RAS", "DeepONet", "PI-DeepONet"]
SAVE_FIGURES = True
SHOW_PER_BENCHMARK_FIGURES = False  # All figures are saved; toggle True for inline display.
RUN_INVERSE_EXTENSION = True

print(f"Profile: {RUN_PROFILE} | Device: {DEVICE} | Torch: {torch.__version__}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

### Reproducibility and interpretation rules

- **Common state/time scaling.** Each ODE is solved in nondimensional network coordinates \(\tau=t/T\) and \(x=y/s\), with problem-specific time horizon \(T\) and state scale \(s\). The governing equation is transformed consistently before training.
- **Initial conditions.** The forward PINNs and operator models use a hard initial-condition ansatz \(\hat x(\tau)=x_0+\tau N_\theta(\tau)\), following the trial-function idea in [L1]. This isolates equation residual error from an avoidable initial-condition trade-off.
- **Reference solutions.** Where available, the notebook uses analytical solutions. Otherwise it uses tightly-toleranced `solve_ivp` trajectories as a numerical reference; these are evaluation references, not training labels for the PINNs.
- **Fairness.** All forward methods use the same architecture depth/width within a profile, target parameters, evaluation grid, seed, and normalized coordinates. DeepONets receive a family of solved trajectories because that is their intended amortized-learning setting; PI-DeepONet adds an equation residual penalty.
- **Memory.** CPU memory is reported as a conservative process high-water-mark proxy; on GPU, PyTorch allocator peak is additionally reported. Neither is a hardware-independent absolute memory benchmark.
- **Claims.** Quick-profile results are diagnostic. Use the `paper` profile, multiple seeds, confidence intervals, and a frozen environment before treating a ranking as publication evidence.

In [ ]:
# 2. Environment and experimental provenance record
environment = {
    "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": torch.__version__,
    "device": str(DEVICE),
    "run_profile": RUN_PROFILE,
    "seed": SEED,
    "profile_config": PROFILE,
}
display(pd.DataFrame(environment.items(), columns=["field", "value"]))
(OUTPUT_DIR / "environment.json").write_text(json.dumps(environment, indent=2))

## 2. Literature review and survey

### 2.1 Chronology and conceptual map

| Era | Development | Relevance to this study |
|---|---|---|
| 1998 | Neural trial functions that exactly satisfy boundary/initial conditions [L1] | Motivates the hard-IC ansatz used here. |
| 2017–2019 | Continuous-time and discrete-time PINNs combine data, initial/boundary conditions, and differential residuals [L2] | Establishes the baseline formulation. |
| 2020–2022 | Training-pathology analysis, loss balancing, adaptive activations, and residual-aware sampling become central [L5–L11] | Explains why collocation allocation matters. |
| 2020–2023 | Variational, conservative, multi-domain, multi-scale, and finite-basis variants broaden PINN design [L12–L17] | Provides alternatives when one global MLP is inadequate. |
| 2021–present | DeepONet, FNO, PI-DeepONet, PINO, and neural-operator theory shift from single-instance solvers to solution-family learners [L18–L22] | Motivates operator baselines and amortized inference. |
| 2024–2026 | Benchmarks and optimizer/loss-landscape studies emphasize controlled comparisons and method–problem matching [L23–L25] | Motivates this transparent protocol and cautious conclusions. |

### 2.2 Foundational formulations

For a first-order ODE system

\[
\frac{d y(t)}{dt}=f(y(t),t;\lambda),\qquad y(0)=y_0,\qquad t\in[0,T],
\]

a continuous-time PINN represents \(y\) by \(y_\theta(t)\) and forms the residual

\[
r_\theta(t)=\frac{d y_\theta(t)}{dt}-f(y_\theta(t),t;\lambda).
\]

With data \(\{(t_i,y_i)\}\), initial-condition points \(\mathcal I\), and collocation points \(\mathcal C\), the conventional soft-constraint objective is

\[
\mathcal L(\theta,\lambda)=w_r\frac{1}{|\mathcal C|}\sum_{t\in\mathcal C}\|r_\theta(t)\|^2+
w_{ic}\frac{1}{|\mathcal I|}\sum_{t\in\mathcal I}\|y_\theta(t)-y_0\|^2+
w_d\frac{1}{N_d}\sum_i\|y_\theta(t_i)-y_i\|^2.
\]

Continuous-time PINNs differentiate the network directly. Discrete-time PINNs instead embed a time-stepping rule, commonly a Runge–Kutta representation, in the network output/loss [L2]. The latter may be attractive when observations are separated by a large time step; the former is the focus of this notebook because it offers a common residual for all ODE cases.

With the hard ansatz \(x_\theta(\tau)=x_0+\tau N_\theta(\tau)\), \(x_\theta(0)=x_0\) exactly, so the forward objective reduces to the normalized residual loss. This is not universally possible for general boundary conditions, but is natural for the initial-value ODE problems here.

### 2.3 Sampling strategies

| Family | Rule | Strength | Limitation / diagnostic |
|---|---|---|---|
| Uniform grid / random | Fixed points sampled uniformly in the domain | Simple, unbiased coverage; strong baseline | May under-sample sharp or short-lived features. |
| Latin hypercube sampling (LHS) | Stratify each coordinate interval once per sample | Better one-dimensional/low-dimensional space filling than naive random draws | Still non-adaptive; does not see the residual. |
| RAR / RAR-G | Add the highest-residual candidate points after training rounds [L3, L9] | Targets persistent hard regions; keeps points | Can over-focus on transient/noisy residual peaks and costs candidate evaluation. |
| RAS / RAD-style | Resample from a residual-proportional distribution [L9] | Spends a fixed budget where residual mass lies | Point set can forget regions; requires exploration mixture. |
| Importance / error-informed | Sample proportional to an estimate of loss, residual, gradient, or failure probability [L10, L11] | Can reduce estimator variance or focus on difficult zones | The sampling estimate can be biased or expensive. |
| Curriculum / causal / dynamic | Progressively unlock time/complexity or use causal residual weights [L24] | Useful for long horizons, multi-scale or chaotic dynamics | Adds schedule hyperparameters and may not transfer across problems. |

**Implementation mapping.** `PINN-LHS` uses fixed LHS points; `RAR` uses greedy residual refinement; `RAS` uses a mixed residual-proportional distribution with a uniform exploration component. The notebook intentionally exposes the point locations so the adaptive mechanism can be audited.

### 2.4 Advanced PINN variants and neural operators

| Method | Core modification | Typical advantage | Principal trade-off |
|---|---|---|---|
| XPINNs | Couple subdomain networks with interface constraints [L12] | Parallelism; complex/multiphysics domains | Interface and partition design. |
| cPINNs | Enforce flux conservation across discrete subdomains [L13] | Conservation-law structure | Specialized for conservation interfaces. |
| VPINNs / hp-VPINNs | Weak/variational residual with test functions; hp refinement [L14, L15] | Lower derivative order; localized refinement | Quadrature/test-space choices. |
| FBPINNs | Sum local, overlapping, normalized subnetwork bases [L16] | Long domains and multiscale structure | Domain decomposition orchestration. |
| Multi-scale PINNs / Fourier features | Improve representation of high-frequency or multiscale modes [L17] | Mitigates spectral bias in selected settings | Frequencies/scales become tuning choices. |
| Self-adaptive PINNs | Learn pointwise loss weights with attention-like masks [L7] | Emphasizes difficult regions without resampling | Min–max or weight-stability concerns. |
| Gradient-enhanced PINNs | Penalize derivatives of the residual [L8] | Extra local regularity information | Higher AD cost and noise sensitivity. |
| DeepONet | Branch encodes an input function/parameter; trunk encodes query location [L18] | Amortizes many solves | Requires representative family data. |
| PI-DeepONet | Adds physical residual/IC/BC constraints to DeepONet [L19] | Reduces paired-data requirement; improves physical consistency | More expensive training and still a soft-constrained operator model. |
| FNO / PINO | Learn operator updates in Fourier space; add physics constraints for PINO [L20, L21] | Resolution-aware operator learning for gridded fields | Primarily natural for field/grid problems rather than small ODE states. |
| General neural operators | Learn maps between function spaces [L22] | Unifies architectures and analysis | Requires distributional generalization, not only a single solve. |

### 2.5 What this review does—and does not—claim

The literature gives strong reasons to test adaptive and operator approaches, but it does **not** justify a universal winner. An adaptive scheme can reduce error when residual concentration correlates with solution error; it can also reinforce a poor early residual estimate. A DeepONet can make per-query inference cheap after family-level training, but its comparison with a single-instance PINN must distinguish **offline training cost** from **online query cost**. This notebook reports both where possible and separates observations from interpretation.

## 3. Methodology

### 3.1 Nondimensionalization and governing residual

Let \(\tau=t/T\) and \(x=y/s\) (elementwise state scale \(s\)). The ODE becomes

\[
\frac{d x}{d\tau}=g(x;\lambda)=\frac{T}{s}\odot f(s\odot x,T\tau;\lambda).
\]

All models in the forward benchmark approximate \(x(\tau)\) and minimize

\[
\mathcal L_r=\frac{1}{N_c}\sum_{j=1}^{N_c}\left\|\frac{d x_\theta}{d\tau}(\tau_j)-g(x_\theta(\tau_j);\lambda)\right\|_2^2.
\]

For the optional inverse logistic experiment, trainable physical parameters \(\lambda\) are mapped from unconstrained values through a sigmoid into predeclared ranges, and a data term is added. Parameter bounds are a numerical stabilization device; they should be declared before looking at results.

### 3.2 Architectures and optimization

**PINN / adaptive PINN.** A fully connected `tanh` MLP maps a scalar normalized time to the normalized state vector. The default profile uses three hidden layers of width 32; the paper profile uses four layers of width 64. Xavier initialization and Adam are used in both profiles. The paper profile adds a final L-BFGS stage to mirror common PINN practice [L25].

**DeepONet.** The branch network maps normalized ODE parameters \(\lambda\) to a state-specific latent vector; the trunk maps \(\tau\) to the same latent dimension. The output is a state-specific inner product:

\[
\hat x_k(\lambda,\tau)=x_{0,k}+\tau\big(b_k(\lambda)^\top t(\tau)+c_k\big).
\]

The same hard initial-condition construction is used. Standard DeepONet minimizes data loss over parameter–time trajectory samples. PI-DeepONet minimizes \(\mathcal L_{data}+w_r\mathcal L_r\), drawing fresh parameter/time collocation pairs for the physics term.

### 3.3 Complexity expectations

Let \(P\) denote trainable parameters, \(N_c\) collocation points, \(S\) state dimension, \(R\) RAR rounds, \(N_{pool}\) candidate points, and \(p\) latent width. A residual-based PINN training step is approximately \(\mathcal O(N_c P)\) with an automatic-differentiation multiplier that grows with derivative order and state dimension. RAR adds roughly \(\mathcal O(R N_{pool}P)\) residual-evaluation work, while RAS adds repeated candidate residual scoring. DeepONet offline training includes family data generation plus tensor products of branch/trunk features; its online evaluation for a new parameter is inexpensive relative to a re-trained PINN, but only after the offline cost is amortized.

The table below makes the exact hyperparameters visible at execution time; it is not a tuned, method-specific hyperparameter search.

In [ ]:
# 3.1 Display the actual hyperparameters used in this run
hyperparameter_table = pd.DataFrame(
    [(k, v) for k, v in PROFILE.items()], columns=["hyperparameter", "value"]
)
display(hyperparameter_table)

## 4. Unified experimental framework

### 4.1 Provenance from the supplied notebooks

The supplied notebooks use a consistent PyTorch/tanh/automatic-differentiation pattern. This notebook reimplements—not executes in-place—their essential mechanisms so that all methods share one evaluation protocol.

| Supplied notebook(s) | Incorporated contribution | Treatment in this unified notebook |
|---|---|---|
| `01_intro_to_pinn.ipynb` | Scalar exponential-decay PINN and residual autodiff | Forward benchmark: exponential decay. |
| `02_second_order_odes.ipynb` | Damped oscillator and hard IC ansatz | Forward benchmark: damped oscillator. |
| `05_reaction_kinetics.ipynb`, `02_consecutive_reaction_kinetics.ipynb` | Three-state consecutive kinetics, conservation diagnostic | Forward benchmark: A→B→C. |
| `01_logistic_population_growth.ipynb` | Data-informed logistic inverse PINN, RAR, DeepONet | Forward logistic family plus optional inverse extension. |
| `03_lotka_volterra_lynx_hare.ipynb` | Coupled nonlinear inverse dynamics and RAR/DeepONet comparison | Lotka–Volterra forward family; real-data context cataloged. |
| `04_sir_epidemic_model.ipynb` | SIR inverse model and parameterized DeepONet | SIR forward family. |
| `05_van_der_pol_oscillator.ipynb` | Nonlinear oscillator and parameter learning | Van der Pol forward family. |
| `06_stiff_kinetics_qssa_enzyme.ipynb` | QSSA enzyme progress curve | Stiff-like enzyme forward family. |
| `07_bioprocess_viable_cell_density.ipynb` | Monod kinetics and two-state bioprocess | Monod forward family. |
| `08_adaptive_sampling.ipynb` | Boundary-layer RAR demonstration | Sampling-method provenance; described as an optional PDE/BVP extension. |
| `09_deeponet.ipynb` | Branch/trunk antiderivative operator | DeepONet implementation pattern. |
| Heat, Burgers, wave, inverse-heat notebooks | PDE and inverse-PDE pedagogical extensions | Cataloged in scope; not numerically mixed with ODE scores. |

### 4.2 Benchmark definitions

The next cells instantiate nine ODE/system-ODE cases. `solve_ivp` trajectories are generated at high tolerance for evaluation only. Exact formulas are used where available. The benchmark table reports the original physical variables, while training uses the normalized form defined above.

In [ ]:
# 4.3 Benchmark registry: physical ODEs plus normalized torch residuals
@dataclass
class ODEBenchmark:
    key: str
    title: str
    equation: str
    state_names: tuple[str, ...]
    parameter_names: tuple[str, ...]
    theta: np.ndarray
    parameter_ranges: np.ndarray  # shape (P, 2)
    y0: np.ndarray
    horizon: float
    state_scale: np.ndarray
    rhs_numpy: object
    rhs_torch: object
    exact_solution: object | None = None
    reference_note: str = "DOP853 solve_ivp"
    source_notebooks: tuple[str, ...] = field(default_factory=tuple)
    challenge: str = ""

    def __post_init__(self):
        self.theta = np.asarray(self.theta, dtype=float)
        self.parameter_ranges = np.asarray(self.parameter_ranges, dtype=float)
        self.y0 = np.asarray(self.y0, dtype=float)
        self.state_scale = np.asarray(self.state_scale, dtype=float)

    @property
    def state_dim(self) -> int:
        return len(self.state_names)

    @property
    def param_dim(self) -> int:
        return len(self.parameter_names)

    @property
    def x0(self) -> np.ndarray:
        return self.y0 / self.state_scale

    def reference(self, n: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        t = np.linspace(0.0, self.horizon, n)
        if self.exact_solution is not None:
            y = np.asarray(self.exact_solution(t, self.theta), dtype=float)
        else:
            sol = solve_ivp(
                lambda time, state: self.rhs_numpy(time, state, self.theta),
                (0.0, self.horizon), self.y0, t_eval=t, method="DOP853",
                rtol=1e-10, atol=1e-12,
            )
            if not sol.success:
                raise RuntimeError(f"Reference solver failed for {self.key}: {sol.message}")
            y = sol.y.T
        return t, t / self.horizon, y / self.state_scale

    def sample_parameters(self, n: int, rng: np.random.Generator) -> np.ndarray:
        lo, hi = self.parameter_ranges[:, 0], self.parameter_ranges[:, 1]
        return rng.uniform(lo, hi, size=(n, self.param_dim))

    def normalize_theta(self, theta: torch.Tensor) -> torch.Tensor:
        lo = torch.as_tensor(self.parameter_ranges[:, 0], device=theta.device, dtype=theta.dtype)
        hi = torch.as_tensor(self.parameter_ranges[:, 1], device=theta.device, dtype=theta.dtype)
        return (theta - lo) / (hi - lo)

    def rhs_tau(self, x: torch.Tensor, theta: torch.Tensor) -> torch.Tensor:
        '''Normalized right-hand side dx/dtau for x=y/state_scale.'''
        scale = torch.as_tensor(self.state_scale, device=x.device, dtype=x.dtype).view(1, -1)
        y = x * scale
        dy_dt = self.rhs_torch(y, theta)
        return self.horizon * dy_dt / scale


def exact_decay(t, theta):
    lam = float(theta[0])
    return np.exp(-lam * t)[:, None]


def exact_damped(t, theta):
    omega0, zeta = map(float, theta)
    wd = omega0 * np.sqrt(1.0 - zeta**2)
    u0, v0 = 1.0, 0.0
    a = (v0 + zeta * omega0 * u0) / wd
    envelope = np.exp(-zeta * omega0 * t)
    core = u0 * np.cos(wd * t) + a * np.sin(wd * t)
    u = envelope * core
    v = envelope * (-zeta * omega0 * core - u0 * wd * np.sin(wd * t) + a * wd * np.cos(wd * t))
    return np.column_stack([u, v])


def exact_logistic(t, theta):
    r, K = map(float, theta)
    y0 = 3.929
    y = K / (1.0 + (K / y0 - 1.0) * np.exp(-r * t))
    return y[:, None]


def exact_reaction(t, theta):
    k1, k2 = map(float, theta)
    A = np.exp(-k1 * t)
    if abs(k2 - k1) < 1e-8:
        B = k1 * t * np.exp(-k1 * t)
    else:
        B = k1 / (k2 - k1) * (np.exp(-k1 * t) - np.exp(-k2 * t))
    C = 1.0 - A - B
    return np.column_stack([A, B, C])


BENCHMARKS = {
    "linear_decay": ODEBenchmark(
        key="linear_decay", title="Exponential decay", equation=r"y'=-\\lambda y,\\ y(0)=1",
        state_names=("y",), parameter_names=("lambda",), theta=np.array([1.0]),
        parameter_ranges=np.array([[0.4, 1.6]]), y0=np.array([1.0]), horizon=5.0,
        state_scale=np.array([1.0]),
        rhs_numpy=lambda t, y, p: np.array([-p[0] * y[0]]),
        rhs_torch=lambda y, p: -p[:, 0:1] * y,
        exact_solution=exact_decay, reference_note="Analytical",
        source_notebooks=("project_sources/05-01_intro_to_pinn.ipynb",),
        challenge="single-scale sanity check",
    ),
    "damped_oscillator": ODEBenchmark(
        key="damped_oscillator", title="Damped harmonic oscillator",
        equation=r"u'=v,\\ v'=-2\\zeta\\omega_0v-\\omega_0^2u",
        state_names=("u", "v"), parameter_names=("omega0", "zeta"), theta=np.array([3.0, 0.20]),
        parameter_ranges=np.array([[2.0, 4.0], [0.10, 0.35]]), y0=np.array([1.0, 0.0]), horizon=6.0,
        state_scale=np.array([1.0, 3.0]),
        rhs_numpy=lambda t, y, p: np.array([y[1], -2*p[1]*p[0]*y[1] - p[0]**2*y[0]]),
        rhs_torch=lambda y, p: torch.cat([y[:, 1:2], -2*p[:, 1:2]*p[:, 0:1]*y[:, 1:2] - p[:, 0:1]**2*y[:, 0:1]], dim=1),
        exact_solution=exact_damped, reference_note="Analytical",
        source_notebooks=("project_sources/04-02_second_order_odes.ipynb",),
        challenge="oscillation and second-order dynamics",
    ),
    "logistic_growth": ODEBenchmark(
        key="logistic_growth", title="Logistic population growth",
        equation=r"P'=rP(1-P/K)", state_names=("P",), parameter_names=("r", "K"),
        theta=np.array([0.0291, 197.0]), parameter_ranges=np.array([[0.010, 0.055], [100.0, 300.0]]),
        y0=np.array([3.929]), horizon=150.0, state_scale=np.array([197.0]),
        rhs_numpy=lambda t, y, p: np.array([p[0] * y[0] * (1 - y[0] / p[1])]),
        rhs_torch=lambda y, p: p[:, 0:1] * y * (1 - y / p[:, 1:2]),
        exact_solution=exact_logistic, reference_note="Analytical",
        source_notebooks=("upload/01_logistic_population_growth.ipynb",),
        challenge="parameterized saturation and sparse-data inverse extension",
    ),
    "consecutive_reaction": ODEBenchmark(
        key="consecutive_reaction", title="Consecutive reaction A→B→C",
        equation=r"A'=-k_1A,\\ B'=k_1A-k_2B,\\ C'=k_2B",
        state_names=("A", "B", "C"), parameter_names=("k1", "k2"), theta=np.array([1.0, 0.6]),
        parameter_ranges=np.array([[0.4, 1.8], [0.25, 1.2]]), y0=np.array([1.0, 0.0, 0.0]), horizon=6.0,
        state_scale=np.array([1.0, 1.0, 1.0]),
        rhs_numpy=lambda t, y, p: np.array([-p[0]*y[0], p[0]*y[0]-p[1]*y[1], p[1]*y[1]]),
        rhs_torch=lambda y, p: torch.cat([-p[:, 0:1]*y[:, 0:1], p[:, 0:1]*y[:, 0:1]-p[:, 1:2]*y[:, 1:2], p[:, 1:2]*y[:, 1:2]], dim=1),
        exact_solution=exact_reaction, reference_note="Analytical",
        source_notebooks=("upload/02_consecutive_reaction_kinetics.ipynb", "project_sources/08-05_reaction_kinetics.ipynb"),
        challenge="coupling, transient intermediate, mass conservation",
    ),
    "lotka_volterra": ODEBenchmark(
        key="lotka_volterra", title="Lotka–Volterra predator–prey",
        equation=r"H'=\\alpha H-\\beta HL,\\ L'=\\delta HL-\\gamma L",
        state_names=("hare", "lynx"), parameter_names=("alpha", "beta", "gamma", "delta"),
        theta=np.array([0.55, 0.028, 0.84, 0.026]),
        parameter_ranges=np.array([[0.30, 0.90], [0.010, 0.050], [0.60, 1.20], [0.010, 0.050]]),
        y0=np.array([30.0, 4.0]), horizon=20.0, state_scale=np.array([100.0, 100.0]),
        rhs_numpy=lambda t, y, p: np.array([p[0]*y[0]-p[1]*y[0]*y[1], p[3]*y[0]*y[1]-p[2]*y[1]]),
        rhs_torch=lambda y, p: torch.cat([p[:, 0:1]*y[:, 0:1]-p[:, 1:2]*y[:, 0:1]*y[:, 1:2], p[:, 3:4]*y[:, 0:1]*y[:, 1:2]-p[:, 2:3]*y[:, 1:2]], dim=1),
        source_notebooks=("upload/03_lotka_volterra_lynx_hare.ipynb",),
        challenge="nonlinear coupling and cyclic trajectories",
    ),
    "sir": ODEBenchmark(
        key="sir", title="SIR epidemic model",
        equation=r"S'=-\\beta SI,\\ I'=\\beta SI-\\gamma I,\\ R'=\\gamma I",
        state_names=("S", "I", "R"), parameter_names=("beta", "gamma"), theta=np.array([1.55, 0.45]),
        parameter_ranges=np.array([[0.8, 2.5], [0.2, 0.8]]), y0=np.array([762/763, 1/763, 0.0]), horizon=14.0,
        state_scale=np.array([1.0, 1.0, 1.0]),
        rhs_numpy=lambda t, y, p: np.array([-p[0]*y[0]*y[1], p[0]*y[0]*y[1]-p[1]*y[1], p[1]*y[1]]),
        rhs_torch=lambda y, p: torch.cat([-p[:, 0:1]*y[:, 0:1]*y[:, 1:2], p[:, 0:1]*y[:, 0:1]*y[:, 1:2]-p[:, 1:2]*y[:, 1:2], p[:, 1:2]*y[:, 1:2]], dim=1),
        source_notebooks=("upload/04_sir_epidemic_model.ipynb",),
        challenge="small infected fraction and conservation",
    ),
    "van_der_pol": ODEBenchmark(
        key="van_der_pol", title="Van der Pol oscillator",
        equation=r"x'=v,\\ v'=\\mu(1-x^2)v-x",
        state_names=("x", "v"), parameter_names=("mu",), theta=np.array([1.0]),
        parameter_ranges=np.array([[0.3, 2.5]]), y0=np.array([2.0, 0.0]), horizon=16.0,
        state_scale=np.array([2.0, 3.0]),
        rhs_numpy=lambda t, y, p: np.array([y[1], p[0]*(1-y[0]**2)*y[1]-y[0]]),
        rhs_torch=lambda y, p: torch.cat([y[:, 1:2], p[:, 0:1]*(1-y[:, 0:1]**2)*y[:, 1:2]-y[:, 0:1]], dim=1),
        source_notebooks=("upload/05_van_der_pol_oscillator.ipynb",),
        challenge="nonlinear oscillation and temporal scale separation",
    ),
    "qssa_enzyme": ODEBenchmark(
        key="qssa_enzyme", title="QSSA enzyme progress curve",
        equation=r"S'=-V_{max}S/(K_m+S)", state_names=("S",), parameter_names=("Vmax", "Km"),
        theta=np.array([0.60, 1.006]), parameter_ranges=np.array([[0.20, 1.20], [0.30, 3.00]]),
        y0=np.array([10.0]), horizon=30.0, state_scale=np.array([10.0]),
        rhs_numpy=lambda t, y, p: np.array([-p[0]*y[0]/(p[1]+y[0])]),
        rhs_torch=lambda y, p: -p[:, 0:1]*y/(p[:, 1:2]+y),
        source_notebooks=("upload/06_stiff_kinetics_qssa_enzyme.ipynb",),
        challenge="stiff-like saturation and reduced-mechanism mismatch",
    ),
    "monod_bioprocess": ODEBenchmark(
        key="monod_bioprocess", title="Monod viable-cell-density bioprocess",
        equation=r"X'=\\mu_{max}S/(K_s+S)X,\\ S'=-(1/Y_{X/S})\\mu X",
        state_names=("X", "S"), parameter_names=("mumax", "Ks", "Yxs"), theta=np.array([0.32, 4.0, 0.50]),
        parameter_ranges=np.array([[0.10, 0.50], [1.0, 10.0], [0.30, 0.80]]),
        y0=np.array([0.2, 20.0]), horizon=30.0, state_scale=np.array([1.0, 20.0]),
        rhs_numpy=lambda t, y, p: np.array([(p[0]*y[1]/(p[1]+y[1]))*y[0], -(1/p[2])*(p[0]*y[1]/(p[1]+y[1]))*y[0]]),
        rhs_torch=lambda y, p: torch.cat([(p[:, 0:1]*y[:, 1:2]/(p[:, 1:2]+y[:, 1:2]))*y[:, 0:1], -(1/p[:, 2:3])*(p[:, 0:1]*y[:, 1:2]/(p[:, 1:2]+y[:, 1:2]))*y[:, 0:1]], dim=1),
        source_notebooks=("upload/07_bioprocess_viable_cell_density.ipynb",),
        challenge="coupled growth/substrate depletion and scale imbalance",
    ),
}

benchmark_rows = []
for bm in BENCHMARKS.values():
    benchmark_rows.append({
        "benchmark": bm.title,
        "states": ", ".join(bm.state_names),
        "parameters": ", ".join(bm.parameter_names),
        "horizon": bm.horizon,
        "reference": bm.reference_note,
        "challenge": bm.challenge,
    })
benchmark_table = pd.DataFrame(benchmark_rows)
display(benchmark_table)

In [ ]:
# 4.4 Optional source-file inventory: informative only, never a runtime dependency.
SOURCE_PATHS = [
    "upload/01_logistic_population_growth.ipynb", "upload/02_consecutive_reaction_kinetics.ipynb",
    "upload/03_lotka_volterra_lynx_hare.ipynb", "upload/04_sir_epidemic_model.ipynb",
    "upload/05_van_der_pol_oscillator.ipynb", "upload/06_stiff_kinetics_qssa_enzyme.ipynb",
    "upload/07_bioprocess_viable_cell_density.ipynb", "project_sources/05-01_intro_to_pinn.ipynb",
    "project_sources/04-02_second_order_odes.ipynb", "project_sources/08-05_reaction_kinetics.ipynb",
    "project_sources/03-08_adaptive_sampling.ipynb", "project_sources/02-09_deeponet.ipynb",
    "project_sources/01-07_inverse_parameter_estimation.ipynb", "project_sources/06-04_burgers_equation.ipynb",
    "project_sources/07-03_heat_equation.ipynb", "project_sources/09-06_wave_equation.ipynb",
]
inventory = pd.DataFrame({"supplied_path": SOURCE_PATHS, "present_in_current_workspace": [Path(p).exists() for p in SOURCE_PATHS]})
display(inventory)

In [ ]:
# 4.5 Visual sanity check of reference solutions before any neural training
fig, axes = plt.subplots(3, 3, figsize=(15, 11), constrained_layout=True)
for ax, bm in zip(axes.ravel(), BENCHMARKS.values()):
    t, tau, x_ref = bm.reference(241)
    for j, name in enumerate(bm.state_names):
        ax.plot(t, x_ref[:, j] * bm.state_scale[j], lw=2, label=name)
    ax.set_title(bm.title)
    ax.set_xlabel("physical time")
    ax.set_ylabel("physical state")
    ax.legend(fontsize=8, ncol=min(3, bm.state_dim))
fig.suptitle("Reference trajectories used only for evaluation", y=1.02, fontsize=15, fontweight="bold")
if SAVE_FIGURES:
    fig.savefig(FIG_DIR / "reference_trajectories.png", bbox_inches="tight")
plt.show()

## 5. Implementation

The code below is deliberately compact but complete. It uses:

- a shared hard-IC MLP for `PINN-LHS`, `RAR`, and `RAS`;
- automatic differentiation for all time derivatives;
- a greedy RAR loop that **adds** high-residual candidate points;
- a residual-adaptive sampler that **resamples** a fixed-sized set with a uniform exploration mixture;
- a vector-valued DeepONet with a state-specific branch latent vector;
- a PI-DeepONet residual term at newly sampled parameter–time points;
- a numerical reference that never enters a forward-PINN loss.

The implementation intentionally does not include every reviewed variant. XPINNs, cPINNs, VPINNs, hp-VPINNs, FBPINNs, FNO, and PINO belong in a larger PDE/domain-decomposition study and are discussed in the review, not misrepresented as implemented ODE baselines here.

In [ ]:
# 5.1 Shared networks and automatic-differentiation residual utilities
def make_mlp(in_dim: int, out_dim: int, width: int, depth: int) -> nn.Module:
    layers: list[nn.Module] = []
    previous = in_dim
    for _ in range(depth):
        layers.extend([nn.Linear(previous, width), nn.Tanh()])
        previous = width
    layers.append(nn.Linear(previous, out_dim))
    net = nn.Sequential(*layers)
    for module in net.modules():
        if isinstance(module, nn.Linear):
            nn.init.xavier_normal_(module.weight)
            nn.init.zeros_(module.bias)
    return net


class HardICPINN(nn.Module):
    '''x(tau) = x0 + tau * N(tau), enforcing x(0)=x0 exactly.'''
    def __init__(self, x0: np.ndarray, width: int, depth: int):
        super().__init__()
        self.core = make_mlp(1, len(x0), width, depth)
        self.register_buffer("x0", torch.tensor(x0, dtype=DTYPE).view(1, -1))

    def forward(self, tau: torch.Tensor) -> torch.Tensor:
        return self.x0 + tau * self.core(tau)


def time_derivative(values: torch.Tensor, tau: torch.Tensor) -> torch.Tensor:
    '''Return componentwise d(values)/d(tau) for batched pointwise networks.'''
    components = []
    for j in range(values.shape[1]):
        derivative = torch.autograd.grad(
            values[:, j:j+1], tau, grad_outputs=torch.ones_like(values[:, j:j+1]),
            create_graph=True, retain_graph=True,
        )[0]
        components.append(derivative)
    return torch.cat(components, dim=1)


def residual_for_model(model: nn.Module, bm: ODEBenchmark, tau: torch.Tensor, theta: torch.Tensor | None = None):
    tau = tau.detach().clone().to(DEVICE, dtype=DTYPE).requires_grad_(True)
    x = model(tau)
    dx_dtau = time_derivative(x, tau)
    if theta is None:
        theta = torch.tensor(bm.theta, device=DEVICE, dtype=DTYPE).view(1, -1).repeat(len(tau), 1)
    elif theta.ndim == 1:
        theta = theta.view(1, -1).repeat(len(tau), 1)
    return dx_dtau - bm.rhs_tau(x, theta), x, tau


def residual_mse(model: nn.Module, bm: ODEBenchmark, tau: torch.Tensor, theta: torch.Tensor | None = None) -> torch.Tensor:
    residual, _, _ = residual_for_model(model, bm, tau, theta)
    return torch.mean(residual.square())


def latin_hypercube(n: int, seed: int) -> torch.Tensor:
    sample = qmc.LatinHypercube(d=1, seed=seed).random(n=n)
    return torch.tensor(sample, dtype=DTYPE, device=DEVICE)


def uniform_samples(n: int, generator: torch.Generator) -> torch.Tensor:
    return torch.rand((n, 1), device=DEVICE, dtype=DTYPE, generator=generator)


def process_memory_mb() -> float:
    # ru_maxrss is KiB on Linux and bytes on macOS; use it as a conservative proxy.
    try:
        if resource is None:
            return float("nan")
        value = float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
        return value / (1024.0 if sys.platform != "darwin" else 1024.0**2)
    except Exception:
        return float("nan")


def metrics_from_prediction(pred: np.ndarray, truth: np.ndarray) -> dict:
    err = pred - truth
    return {
        "mse": float(np.mean(err**2)),
        "rmse": float(np.sqrt(np.mean(err**2))),
        "relative_l2": float(np.linalg.norm(err.ravel()) / (np.linalg.norm(truth.ravel()) + 1e-12)),
        "max_abs_error": float(np.max(np.abs(err))),
    }

In [ ]:
# 5.2 Adaptive-sampling routines
def greedy_rar_points(model: nn.Module, bm: ODEBenchmark, existing: torch.Tensor, n_add: int, pool_size: int, generator: torch.Generator) -> torch.Tensor:
    candidates = uniform_samples(pool_size, generator)
    residual, _, _ = residual_for_model(model, bm, candidates)
    score = residual.detach().square().sum(dim=1)
    top = torch.topk(score, k=min(n_add, pool_size), largest=True).indices
    new_points = candidates[top].detach()
    return torch.cat([existing.detach(), new_points], dim=0)


def residual_adaptive_points(model: nn.Module, bm: ODEBenchmark, n_points: int, pool_size: int, mix_uniform: float, generator: torch.Generator) -> torch.Tensor:
    '''Residual-proportional resampling with explicit uniform exploration.'''
    candidates = uniform_samples(pool_size, generator)
    residual, _, _ = residual_for_model(model, bm, candidates)
    score = residual.detach().square().sum(dim=1).clamp_min(1e-12)
    probabilities = score / score.sum()
    n_adaptive = int(round(n_points * (1.0 - mix_uniform)))
    n_uniform = n_points - n_adaptive
    adaptive_idx = torch.multinomial(probabilities, num_samples=n_adaptive, replacement=True, generator=generator)
    adaptive = candidates[adaptive_idx].detach()
    exploration = uniform_samples(n_uniform, generator).detach()
    return torch.cat([adaptive, exploration], dim=0)


def train_pinn(bm: ODEBenchmark, method: str, seed_offset: int = 0) -> dict:
    '''Train one standardized forward PINN/adaptive PINN and return audit artifacts.'''
    assert method in {"PINN-LHS", "RAR", "RAS"}
    seed_everything(SEED + seed_offset)
    generator = torch.Generator(device=DEVICE).manual_seed(SEED + seed_offset)
    model = HardICPINN(bm.x0, PROFILE["width"], PROFILE["depth"]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
    history, residual_history, point_history = [], [], []

    if method == "PINN-LHS":
        collocation = latin_hypercube(PROFILE["n_collocation"], SEED + seed_offset)
        schedule = [(PROFILE["pinn_epochs"], False)]
    elif method == "RAR":
        collocation = latin_hypercube(PROFILE["rar_initial"], SEED + seed_offset)
        schedule = [(PROFILE["rar_epochs"], True)] * PROFILE["rar_rounds"]
    else:
        collocation = latin_hypercube(PROFILE["n_collocation"], SEED + seed_offset)
        schedule = [(PROFILE["pinn_epochs"], False)]

    start = perf_counter()
    total_steps = 0
    for round_index, (rounds_epochs, refine_after_round) in enumerate(schedule):
        for _ in range(rounds_epochs):
            if method == "RAS" and total_steps > 0 and total_steps % PROFILE["ras_resample_every"] == 0:
                collocation = residual_adaptive_points(
                    model, bm, PROFILE["n_collocation"], PROFILE["candidate_pool"],
                    PROFILE["ras_mix_uniform"], generator,
                )
            optimizer.zero_grad(set_to_none=True)
            loss = residual_mse(model, bm, collocation)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=100.0)
            optimizer.step()
            total_steps += 1
            if total_steps % PROFILE["log_every"] == 0 or total_steps == 1:
                history.append((total_steps, float(loss.detach().cpu())))
                diagnostic = uniform_samples(min(192, PROFILE["candidate_pool"]), generator)
                residual_history.append((total_steps, float(residual_mse(model, bm, diagnostic).detach().cpu())))
        point_history.append(collocation.detach().cpu().numpy().ravel().copy())
        if refine_after_round and round_index < len(schedule) - 1:
            collocation = greedy_rar_points(
                model, bm, collocation, PROFILE["rar_add"], PROFILE["candidate_pool"], generator
            )

    if PROFILE["lbfgs_max_iter"]:
        lbfgs = torch.optim.LBFGS(model.parameters(), max_iter=PROFILE["lbfgs_max_iter"], line_search_fn="strong_wolfe")
        def closure():
            lbfgs.zero_grad(set_to_none=True)
            value = residual_mse(model, bm, collocation)
            value.backward()
            return value
        lbfgs.step(closure)

    elapsed = perf_counter() - start
    t_eval, tau_eval, x_ref = bm.reference(PROFILE["eval_points"])
    tau_tensor = torch.tensor(tau_eval[:, None], dtype=DTYPE, device=DEVICE)
    with torch.no_grad():
        x_pred = model(tau_tensor).cpu().numpy()
    final_residual = float(residual_mse(model, bm, tau_tensor).detach().cpu())
    memory_gpu = float(torch.cuda.max_memory_allocated(DEVICE) / 1024**2) if DEVICE.type == "cuda" else np.nan
    return {
        "method": method, "benchmark": bm.key, "model": model, "time": t_eval, "tau": tau_eval,
        "prediction": x_pred, "reference": x_ref, "history": history, "residual_history": residual_history,
        "sampling_history": point_history, "final_points": collocation.detach().cpu().numpy().ravel(),
        "training_seconds": elapsed, "iterations": total_steps, "final_residual": final_residual,
        "cpu_memory_proxy_mb": process_memory_mb(), "gpu_peak_mb": memory_gpu,
        **metrics_from_prediction(x_pred, x_ref),
    }

In [ ]:
# 5.3 DeepONet and physics-informed DeepONet
class VectorDeepONet(nn.Module):
    def __init__(self, param_dim: int, state_dim: int, x0: np.ndarray, width: int, latent: int):
        super().__init__()
        self.state_dim = state_dim
        self.latent = latent
        self.branch = make_mlp(param_dim, state_dim * latent, width, 3)
        self.trunk = make_mlp(1, latent, width, 3)
        self.bias = nn.Parameter(torch.zeros(state_dim))
        self.register_buffer("x0", torch.tensor(x0, dtype=DTYPE).view(1, state_dim))

    def forward_pairs(self, theta_normalized: torch.Tensor, tau: torch.Tensor) -> torch.Tensor:
        branch = self.branch(theta_normalized).view(-1, self.state_dim, self.latent)
        trunk = self.trunk(tau)
        raw = torch.einsum("bsp,bp->bs", branch, trunk) + self.bias
        return self.x0 + tau * raw

    def forward_grid(self, theta_normalized: torch.Tensor, tau: torch.Tensor) -> torch.Tensor:
        branch = self.branch(theta_normalized).view(-1, self.state_dim, self.latent)
        trunk = self.trunk(tau)
        raw = torch.einsum("bsp,qp->bqs", branch, trunk) + self.bias.view(1, 1, -1)
        return self.x0.view(1, 1, -1) + tau.view(1, -1, 1) * raw


def make_operator_dataset(bm: ODEBenchmark, n_params: int, n_times: int, seed_offset: int) -> dict:
    rng = np.random.default_rng(SEED + seed_offset)
    theta = bm.sample_parameters(n_params, rng)
    tau = np.linspace(0.0, 1.0, n_times)
    trajectories = []
    for parameter in theta:
        if bm.exact_solution is not None:
            y = bm.exact_solution(tau * bm.horizon, parameter)
        else:
            sol = solve_ivp(
                lambda time, state, parameter=parameter: bm.rhs_numpy(time, state, parameter),
                (0.0, bm.horizon), bm.y0, t_eval=tau * bm.horizon,
                method="DOP853", rtol=1e-9, atol=1e-11,
            )
            if not sol.success:
                raise RuntimeError(f"Operator-data reference solver failed for {bm.key}")
            y = sol.y.T
        trajectories.append(y / bm.state_scale)
    return {
        "theta": torch.tensor(theta, dtype=DTYPE, device=DEVICE),
        "tau": torch.tensor(tau[:, None], dtype=DTYPE, device=DEVICE),
        "x": torch.tensor(np.stack(trajectories), dtype=DTYPE, device=DEVICE),
    }


def pi_residual_loss(model: VectorDeepONet, bm: ODEBenchmark, n_parameter: int, n_time: int, generator: torch.Generator) -> torch.Tensor:
    rng = np.random.default_rng(int(torch.randint(0, 2**31 - 1, (1,), generator=generator, device=DEVICE).item()))
    theta_np = bm.sample_parameters(n_parameter, rng)
    theta = torch.tensor(theta_np, dtype=DTYPE, device=DEVICE)
    theta_norm = bm.normalize_theta(theta)
    tau = torch.rand((n_parameter * n_time, 1), dtype=DTYPE, device=DEVICE, generator=generator).requires_grad_(True)
    theta_pair = theta.repeat_interleave(n_time, dim=0)
    theta_norm_pair = theta_norm.repeat_interleave(n_time, dim=0)
    x = model.forward_pairs(theta_norm_pair, tau)
    dx = time_derivative(x, tau)
    residual = dx - bm.rhs_tau(x, theta_pair)
    return torch.mean(residual.square())


def train_deeponet(bm: ODEBenchmark, physics_informed: bool, seed_offset: int = 0) -> dict:
    seed_everything(SEED + seed_offset)
    generator = torch.Generator(device=DEVICE).manual_seed(SEED + seed_offset)
    dataset = make_operator_dataset(bm, PROFILE["operator_train_params"], PROFILE["operator_train_times"], seed_offset)
    model = VectorDeepONet(bm.param_dim, bm.state_dim, bm.x0, PROFILE["operator_width"], PROFILE["operator_latent"]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
    history, residual_history = [], []
    epochs = PROFILE["pi_deeponet_epochs"] if physics_informed else PROFILE["deeponet_epochs"]
    start = perf_counter()

    for epoch in range(1, epochs + 1):
        parameter_idx = torch.randint(0, len(dataset["theta"]), (PROFILE["operator_batch_params"],), device=DEVICE, generator=generator)
        time_idx = torch.randint(0, len(dataset["tau"]), (PROFILE["operator_batch_times"],), device=DEVICE, generator=generator)
        theta_batch = dataset["theta"][parameter_idx]
        tau_batch = dataset["tau"][time_idx]
        target = dataset["x"][parameter_idx][:, time_idx, :]
        prediction = model.forward_grid(bm.normalize_theta(theta_batch), tau_batch)
        data_loss = torch.mean((prediction - target).square())
        physics_loss = torch.tensor(0.0, device=DEVICE)
        if physics_informed:
            physics_loss = pi_residual_loss(
                model, bm, PROFILE["pi_collocation_params"], PROFILE["pi_collocation_times"], generator
            )
        loss = data_loss + (PROFILE["pi_physics_weight"] * physics_loss if physics_informed else 0.0)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=100.0)
        optimizer.step()
        if epoch % PROFILE["log_every"] == 0 or epoch == 1:
            history.append((epoch, float(loss.detach().cpu())))
            if physics_informed:
                residual_history.append((epoch, float(physics_loss.detach().cpu())))

    elapsed = perf_counter() - start
    t_eval, tau_eval, x_ref = bm.reference(PROFILE["eval_points"])
    theta_target = torch.tensor(bm.theta[None, :], dtype=DTYPE, device=DEVICE)
    tau_tensor = torch.tensor(tau_eval[:, None], dtype=DTYPE, device=DEVICE)
    with torch.no_grad():
        x_pred = model.forward_grid(bm.normalize_theta(theta_target), tau_tensor)[0].cpu().numpy()
    diagnostic_tau = torch.tensor(tau_eval[:, None], dtype=DTYPE, device=DEVICE)
    final_residual = float(pi_residual_loss(model, bm, n_parameter=1, n_time=min(24, len(tau_eval)), generator=generator).detach().cpu())
    method = "PI-DeepONet" if physics_informed else "DeepONet"
    memory_gpu = float(torch.cuda.max_memory_allocated(DEVICE) / 1024**2) if DEVICE.type == "cuda" else np.nan
    return {
        "method": method, "benchmark": bm.key, "model": model, "time": t_eval, "tau": tau_eval,
        "prediction": x_pred, "reference": x_ref, "history": history, "residual_history": residual_history,
        "sampling_history": [], "final_points": np.array([]), "training_seconds": elapsed,
        "iterations": epochs, "final_residual": final_residual, "cpu_memory_proxy_mb": process_memory_mb(),
        "gpu_peak_mb": memory_gpu, "operator_training_parameter_trajectories": PROFILE["operator_train_params"],
        "operator_training_time_nodes": PROFILE["operator_train_times"],
        **metrics_from_prediction(x_pred, x_ref),
    }

### 5.4 Benchmark execution protocol

The following cell runs all five methods on all nine ODE benchmarks in the selected profile. The `quick` profile is deterministic and deliberately small enough for exploratory CPU execution; it should be treated as a smoke-tested computational study, not final hyperparameter-tuned evidence. The `paper` profile gives each method materially larger budgets and a final L-BFGS stage for PINNs.

DeepONet results include offline trajectory-generation/training time. A downstream deployment should separately time cached **online** evaluation; the study therefore avoids interpreting total offline cost as per-query cost.

In [ ]:
# 5.4 Run the unified study. This is the only long-running cell.
all_runs: list[dict] = []
for benchmark_index, bm in enumerate(BENCHMARKS.values()):
    print(f"\n{'='*86}\nBenchmark {benchmark_index + 1}/{len(BENCHMARKS)}: {bm.title}\n{'='*86}")
    method_specs = [
        ("PINN-LHS", lambda bm=bm, offset=benchmark_index*100 + 1: train_pinn(bm, "PINN-LHS", offset)),
        ("RAR", lambda bm=bm, offset=benchmark_index*100 + 2: train_pinn(bm, "RAR", offset)),
        ("RAS", lambda bm=bm, offset=benchmark_index*100 + 3: train_pinn(bm, "RAS", offset)),
        ("DeepONet", lambda bm=bm, offset=benchmark_index*100 + 4: train_deeponet(bm, False, offset)),
        ("PI-DeepONet", lambda bm=bm, offset=benchmark_index*100 + 5: train_deeponet(bm, True, offset)),
    ]
    for method, runner in method_specs:
        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats(DEVICE)
        result = runner()
        all_runs.append(result)
        print(f"{method:14s}  rel-L2={result['relative_l2']:.3e}  RMSE={result['rmse']:.3e}  "
              f"time={result['training_seconds']:.2f}s  residual={result['final_residual']:.3e}")

print(f"\nCompleted {len(all_runs)} method–benchmark runs.")

In [ ]:
# 6. Results table and safe machine-readable export
results = pd.DataFrame([
    {k: v for k, v in run.items() if k not in {"model", "time", "tau", "prediction", "reference", "history", "residual_history", "sampling_history", "final_points"}}
    for run in all_runs
])
results["benchmark_title"] = results["benchmark"].map({key: bm.title for key, bm in BENCHMARKS.items()})
results["log10_relative_l2"] = np.log10(results["relative_l2"].clip(lower=1e-16))
results = results.sort_values(["benchmark", "relative_l2", "training_seconds"]).reset_index(drop=True)
display(results[["benchmark_title", "method", "mse", "relative_l2", "rmse", "max_abs_error", "training_seconds", "iterations", "final_residual", "cpu_memory_proxy_mb"]].style.format({
    "mse": "{:.3e}", "relative_l2": "{:.3e}", "rmse": "{:.3e}", "max_abs_error": "{:.3e}",
    "training_seconds": "{:.2f}", "final_residual": "{:.3e}", "cpu_memory_proxy_mb": "{:.1f}",
}))
results.to_csv(OUTPUT_DIR / "benchmark_results.csv", index=False)

In [ ]:
# 6.1 Publication-style summary figures: accuracy, efficiency, and method-by-problem heatmap
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), constrained_layout=True)

error_order = results.groupby("method")["relative_l2"].median().sort_values().index
sns.boxplot(data=results, x="method", y="relative_l2", order=error_order, palette=[COLORS[m] for m in error_order], ax=axes[0])
axes[0].set_yscale("log")
axes[0].set_title("Relative L2 error across ODE benchmarks")
axes[0].set_xlabel("")
axes[0].set_ylabel("relative L2 error (log scale)")
axes[0].tick_params(axis="x", rotation=30)

for method, group in results.groupby("method"):
    axes[1].scatter(group["training_seconds"], group["relative_l2"], s=70, alpha=0.82, color=COLORS[method], label=method, edgecolor="white", linewidth=0.7)
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_title("Accuracy–training-time trade-off")
axes[1].set_xlabel("offline training time (s, log scale)")
axes[1].set_ylabel("relative L2 error (log scale)")
axes[1].legend(fontsize=8, loc="best")

heat = results.pivot(index="benchmark_title", columns="method", values="log10_relative_l2")
heat = heat.reindex(columns=ACTIVE_METHODS)
sns.heatmap(heat, cmap="mako_r", annot=True, fmt=".2f", linewidths=0.5, cbar_kws={"label": "log10(relative L2 error)"}, ax=axes[2])
axes[2].set_title("Accuracy map (lower is better)")
axes[2].set_xlabel("")
axes[2].set_ylabel("")

if SAVE_FIGURES:
    fig.savefig(FIG_DIR / "study_summary_accuracy_efficiency.png", bbox_inches="tight")
plt.show()

In [ ]:
# 6.2 Per-benchmark figures: solution, error, loss/residual evolution, and sampling distributions
run_lookup = {(run["benchmark"], run["method"]): run for run in all_runs}

for bm in BENCHMARKS.values():
    fig, axes = plt.subplots(2, 2, figsize=(13.5, 8.5), constrained_layout=True)
    reference_run = run_lookup[(bm.key, "PINN-LHS")]
    time = reference_run["time"]
    x_ref = reference_run["reference"]

    # Solution comparison: first state is shown in physical units for readability.
    axes[0, 0].plot(time, x_ref[:, 0] * bm.state_scale[0], color=COLORS["Exact / reference"], lw=2.8, label="reference")
    for method in ACTIVE_METHODS:
        run = run_lookup[(bm.key, method)]
        axes[0, 0].plot(time, run["prediction"][:, 0] * bm.state_scale[0], color=COLORS[method], lw=1.8, alpha=0.92, label=method)
    axes[0, 0].set_title(f"{bm.title}: first state")
    axes[0, 0].set_xlabel("physical time")
    axes[0, 0].set_ylabel(bm.state_names[0])
    axes[0, 0].legend(fontsize=8, ncol=2)

    # Error distribution along time; all states normalized equally by the benchmark scaling.
    for method in ACTIVE_METHODS:
        run = run_lookup[(bm.key, method)]
        abs_error = np.linalg.norm(run["prediction"] - x_ref, axis=1)
        axes[0, 1].plot(time, abs_error, color=COLORS[method], lw=1.8, label=method)
    axes[0, 1].set_yscale("log")
    axes[0, 1].set_title("Pointwise state-vector error")
    axes[0, 1].set_xlabel("physical time")
    axes[0, 1].set_ylabel("normalized L2 error (log scale)")

    # Training-loss and physics-residual histories.
    for method in ACTIVE_METHODS:
        run = run_lookup[(bm.key, method)]
        if run["history"]:
            step, value = np.asarray(run["history"]).T
            axes[1, 0].plot(step, value, color=COLORS[method], lw=1.6, label=method)
        if run["residual_history"]:
            step, value = np.asarray(run["residual_history"]).T
            axes[1, 1].plot(step, value, color=COLORS[method], lw=1.6, label=method)
    axes[1, 0].set_yscale("log")
    axes[1, 0].set_title("Recorded training objective")
    axes[1, 0].set_xlabel("iteration")
    axes[1, 0].set_ylabel("loss (log scale)")
    axes[1, 0].legend(fontsize=8)
    axes[1, 1].set_yscale("log")
    axes[1, 1].set_title("Recorded physics-residual diagnostic")
    axes[1, 1].set_xlabel("iteration")
    axes[1, 1].set_ylabel("residual MSE (log scale)")
    if not axes[1, 1].lines:
        axes[1, 1].text(0.5, 0.5, "No separately logged\nresidual history for data-only DeepONet", ha="center", va="center", transform=axes[1, 1].transAxes)

    # Sampling distribution is saved in a separate compact companion figure.
    if SAVE_FIGURES:
        fig.savefig(FIG_DIR / f"{bm.key}_solutions_errors_training.png", bbox_inches="tight")
    if SHOW_PER_BENCHMARK_FIGURES:
        plt.show()
    else:
        plt.close(fig)

    fig_sampling, ax = plt.subplots(figsize=(8.8, 3.8), constrained_layout=True)
    bins = np.linspace(0, 1, 26)
    for method in ["PINN-LHS", "RAR", "RAS"]:
        points = run_lookup[(bm.key, method)]["final_points"]
        ax.hist(points, bins=bins, density=True, histtype="step", linewidth=2.0, color=COLORS[method], label=f"{method} (n={len(points)})")
    ax.set_title(f"{bm.title}: final collocation-point distribution")
    ax.set_xlabel("normalized time τ")
    ax.set_ylabel("density")
    ax.legend(fontsize=8)
    if SAVE_FIGURES:
        fig_sampling.savefig(FIG_DIR / f"{bm.key}_sampling_distribution.png", bbox_inches="tight")
    if SHOW_PER_BENCHMARK_FIGURES:
        plt.show()
    else:
        plt.close(fig_sampling)

print(f"Saved {len(BENCHMARKS) * 2 + 2} study figures in {FIG_DIR.resolve()}")

In [ ]:
# 6.3 Statistical aggregation and result-grounded comparisons
summary = (results.groupby("method")
           .agg(
               median_relative_l2=("relative_l2", "median"),
               mean_relative_l2=("relative_l2", "mean"),
               median_rmse=("rmse", "median"),
               median_time_s=("training_seconds", "median"),
               median_final_residual=("final_residual", "median"),
               wins=("relative_l2", lambda s: 0),
           ))
winners = results.loc[results.groupby("benchmark")["relative_l2"].idxmin(), ["benchmark", "method"]]
win_counts = winners["method"].value_counts()
summary["wins"] = summary.index.map(win_counts).fillna(0).astype(int)
summary = summary.sort_values("median_relative_l2")
display(summary.style.format({
    "median_relative_l2": "{:.3e}", "mean_relative_l2": "{:.3e}", "median_rmse": "{:.3e}",
    "median_time_s": "{:.2f}", "median_final_residual": "{:.3e}",
}))

comparison_table = results[["benchmark_title", "method", "relative_l2", "training_seconds", "iterations", "final_residual"]].copy()
comparison_table["stability"] = "single deterministic seed; repeat before ranking"
comparison_table["scalability"] = comparison_table["method"].map({
    "PINN-LHS": "per-instance optimization", "RAR": "per-instance + candidate scoring",
    "RAS": "per-instance + resampling", "DeepONet": "amortized after offline training",
    "PI-DeepONet": "amortized, higher offline AD cost",
})
comparison_table["comments"] = comparison_table["method"].map({
    "PINN-LHS": "fixed LHS collocation baseline", "RAR": "greedily retains high-residual points",
    "RAS": "residual-proportional resampling + exploration", "DeepONet": "supervised operator trajectory data",
    "PI-DeepONet": "data + physics operator loss",
})
comparison_table.to_csv(OUTPUT_DIR / "comparative_analysis_table.csv", index=False)
display(comparison_table.head(15).style.format({"relative_l2": "{:.3e}", "training_seconds": "{:.2f}", "final_residual": "{:.3e}"}))

## 7. Comparative analysis

The main comparison table has now been computed above with accuracy, training time, iterations, residual, stability caveat, scalability description, and method-specific comments for every method–benchmark run. Interpret it using the following distinctions:

1. **PINN vs adaptive PINN.** Compare `PINN-LHS` against `RAR`/`RAS` within the same row group. An adaptive benefit is credible only when its error improves without a disproportionate time/candidate-pool cost and the point distribution offers a plausible diagnostic.
2. **PINN vs DeepONet.** A PINN solves one fixed parameter instance, while DeepONet pays an offline family-training cost. DeepONet’s operational benefit is expected only when many new parameter queries use the learned family.
3. **Adaptive PINN vs DeepONet.** These optimize different levers—where to enforce physics for one instance versus whether to amortize across a family—so a single scalar rank is not a complete decision rule.
4. **All methods.** The heatmap and accuracy–time scatter expose problem dependence, rather than hiding it in a global average.

In [ ]:
# 7.1 Render an execution-grounded textual readout (no prewritten numerical claims)
best_method = summary.index[0]
fastest_method = results.groupby("method")["training_seconds"].median().idxmin()
adaptive_rows = results[results["method"].isin(["RAR", "RAS"])].merge(
    results[results["method"] == "PINN-LHS"][["benchmark", "relative_l2"]], on="benchmark", suffixes=("", "_lhs")
)
adaptive_improvement_fraction = float((adaptive_rows["relative_l2"] < adaptive_rows["relative_l2_lhs"]).mean())
readout = f'''
### Executed-study readout

- Lowest median relative-L2 error in this **single-seed, {RUN_PROFILE}-profile** run: **{best_method}**.
- Lowest median offline training time: **{fastest_method}**.
- Adaptive runs (RAR or RAS) beat the LHS PINN on **{adaptive_improvement_fraction:.0%}** of like-for-like benchmark comparisons.
- Method wins by benchmark: **{', '.join(f'{m}: {int(c)}' for m, c in win_counts.items())}**.

These are descriptive results from this configuration, not statistically established universal rankings. Inspect the saved figures and re-run multiple seeds before using them in a paper claim.
'''
display(Markdown(readout))

## 8. Optional data-informed inverse-problem extension

The source notebooks emphasize inverse parameter estimation. To preserve that research dimension without conflating it with the forward-solver table, the next cell fits the logistic-growth parameters \(r\) and \(K\) to the U.S. population series embedded in the supplied logistic notebook. It uses the same hard-IC network and an explicit data-plus-physics loss. The empirical logistic parameters are a modeling approximation, not immutable ground truth; the cell therefore reports fit diagnostics rather than an unsupported recovery claim.

In [ ]:
# 8.1 Data-informed inverse PINN: logistic growth (optional but enabled by default)
if RUN_INVERSE_EXTENSION:
    bm = BENCHMARKS["logistic_growth"]
    years = np.array([1790, 1800, 1810, 1820, 1830, 1840, 1850, 1860, 1870, 1880, 1890, 1900, 1910, 1920, 1930, 1940], dtype=float)
    population = np.array([3.929, 5.308, 7.240, 9.638, 12.866, 17.069, 23.192, 31.443, 38.558, 50.189, 62.948, 75.995, 91.972, 105.711, 122.775, 131.669], dtype=float)
    tau_data = torch.tensor(((years - years[0]) / bm.horizon)[:, None], dtype=DTYPE, device=DEVICE)
    x_data = torch.tensor((population / bm.state_scale[0])[:, None], dtype=DTYPE, device=DEVICE)
    seed_everything(SEED + 919)
    inverse_model = HardICPINN(bm.x0, PROFILE["width"], PROFILE["depth"]).to(DEVICE)
    raw_theta = nn.Parameter(torch.zeros(bm.param_dim, device=DEVICE, dtype=DTYPE))
    lo = torch.tensor(bm.parameter_ranges[:, 0], dtype=DTYPE, device=DEVICE)
    hi = torch.tensor(bm.parameter_ranges[:, 1], dtype=DTYPE, device=DEVICE)
    inverse_optimizer = torch.optim.Adam(list(inverse_model.parameters()) + [raw_theta], lr=2e-3)
    inverse_tau = latin_hypercube(max(80, PROFILE["n_collocation"]), SEED + 919)
    inverse_history = []
    inverse_steps = max(300, int(PROFILE["pinn_epochs"] * 1.25))
    for step in range(1, inverse_steps + 1):
        theta_est = lo + (hi - lo) * torch.sigmoid(raw_theta)
        inverse_optimizer.zero_grad(set_to_none=True)
        physics = residual_mse(inverse_model, bm, inverse_tau, theta_est)
        data_loss = torch.mean((inverse_model(tau_data) - x_data).square())
        loss = physics + 10.0 * data_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(inverse_model.parameters()) + [raw_theta], max_norm=100.0)
        inverse_optimizer.step()
        if step % PROFILE["log_every"] == 0 or step == 1:
            inverse_history.append((step, float(loss.detach().cpu()), float(physics.detach().cpu()), float(data_loss.detach().cpu())))

    theta_est = (lo + (hi - lo) * torch.sigmoid(raw_theta)).detach().cpu().numpy()
    t_eval, tau_eval, x_ref = bm.reference(PROFILE["eval_points"])
    with torch.no_grad():
        x_inv = inverse_model(torch.tensor(tau_eval[:, None], dtype=DTYPE, device=DEVICE)).cpu().numpy()
    observation_fit = metrics_from_prediction(inverse_model(tau_data).detach().cpu().numpy(), x_data.detach().cpu().numpy())
    inverse_summary = pd.DataFrame({
        "parameter": bm.parameter_names,
        "estimate": theta_est,
        "reference_surrogate_parameter": bm.theta,
        "absolute_difference": np.abs(theta_est - bm.theta),
    })
    display(inverse_summary.style.format({"estimate": "{:.5g}", "reference_surrogate_parameter": "{:.5g}", "absolute_difference": "{:.5g}"}))
    print("Observation-fit metrics:", {k: f"{v:.3e}" for k, v in observation_fit.items()})

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
    axes[0].scatter(years, population, color="#222222", s=30, zorder=4, label="U.S. census observations")
    axes[0].plot(years[0] + t_eval, x_inv[:, 0] * bm.state_scale[0], color=COLORS["PI-DeepONet"], lw=2.4, label="data-informed PINN")
    axes[0].set_title("Logistic inverse extension")
    axes[0].set_xlabel("year")
    axes[0].set_ylabel("population (same units as supplied notebook)")
    axes[0].legend(fontsize=8)
    inverse_hist = np.asarray(inverse_history)
    axes[1].plot(inverse_hist[:, 0], inverse_hist[:, 1], label="total")
    axes[1].plot(inverse_hist[:, 0], inverse_hist[:, 2], label="physics")
    axes[1].plot(inverse_hist[:, 0], inverse_hist[:, 3], label="data")
    axes[1].set_yscale("log")
    axes[1].set_title("Inverse objective components")
    axes[1].set_xlabel("iteration")
    axes[1].set_ylabel("loss (log scale)")
    axes[1].legend(fontsize=8)
    if SAVE_FIGURES:
        fig.savefig(FIG_DIR / "inverse_logistic_extension.png", bbox_inches="tight")
    plt.show()
    inverse_summary.to_csv(OUTPUT_DIR / "inverse_logistic_parameters.csv", index=False)

## 9. Discussion

### Why performance can differ

1. **Collocation allocation.** In RAR, points accumulate at locations whose residual is currently high; in RAS, the distribution is repeatedly reweighted by residual magnitude. This can help with fast transients, sharp saturation, or difficult oscillatory portions, but it can also make training path-dependent. The sampling plots are essential evidence: an accuracy gain without interpretable point allocation deserves skepticism.
2. **Optimization imbalance.** PINN losses combine terms with different units and curvature. Nondimensionalization and hard ICs remove two common avoidable issues, but they do not eliminate spectral bias or differential-operator conditioning [L5, L6, L25].
3. **Operator amortization.** DeepONet can predict a new trajectory after one branch evaluation and many trunk evaluations, while a PINN ordinarily reoptimizes per parameter instance. Its training set must nevertheless cover the parameter family; extrapolation beyond the range is not established by the present tests.
4. **PI-DeepONet trade-off.** The physics term can constrain an operator between observed solution samples, but its automatic-differentiation cost makes offline training heavier. Its benefit must be judged against both data availability and the operator’s intended query volume.
5. **Failure modes.** Stiff/reduced kinetics, long-horizon nonlinear oscillation, small compartments in SIR, and coupled state-scale disparities are intentionally included because they expose where a single global MLP may be fragile. A small residual does not prove a small solution error, particularly under incomplete constraints or poor conditioning.

### Recommendations for a follow-on study

- Run at least 5–10 seeds per method and report median/IQR or confidence intervals.
- Tune only through a prespecified budget and report candidate-pool/resampling cost explicitly.
- Add conventional solver runtime/error baselines for the target deployment hardware.
- Separate inverse-data fitting error, parameter-identifiability uncertainty, and trajectory error.
- For long, multiscale, or spatially decomposed problems, benchmark FBPINNs, XPINNs/VPINNs, causal training, and architecture/feature modifications rather than assuming sampling alone is sufficient.
- For deployment across many parameter queries, measure full offline cost, online latency, memory, out-of-distribution error, and data-generation cost for neural operators.

In [ ]:
# 9.1 Save a compact, result-grounded study manifest for audit/reuse
study_manifest = {
    "title": "Unified PINN / adaptive sampling / DeepONet ODE study",
    "profile": RUN_PROFILE,
    "seed": SEED,
    "benchmarks": list(BENCHMARKS),
    "methods": ACTIVE_METHODS,
    "n_runs": len(all_runs),
    "best_median_method": str(best_method),
    "fastest_median_method": str(fastest_method),
    "results_csv": str((OUTPUT_DIR / "benchmark_results.csv").resolve()),
    "figures_directory": str(FIG_DIR.resolve()),
    "caveat": "Single-seed profile results are descriptive; repeat/tune before publication claims.",
}
(OUTPUT_DIR / "study_manifest.json").write_text(json.dumps(study_manifest, indent=2))
display(Markdown(f"Study artifacts written to `{OUTPUT_DIR.resolve()}`."))

In [ ]:
# 9.2 Execution checks: fail loudly if a top-to-bottom run produced incomplete evidence
expected_runs = len(BENCHMARKS) * len(ACTIVE_METHODS)
assert len(results) == expected_runs, f"Expected {expected_runs} runs, found {len(results)}"
assert results[["mse", "relative_l2", "rmse", "max_abs_error", "training_seconds", "final_residual"]].apply(np.isfinite).all().all(), "Non-finite metric found"
assert (OUTPUT_DIR / "benchmark_results.csv").exists()
assert (OUTPUT_DIR / "comparative_analysis_table.csv").exists()
assert (OUTPUT_DIR / "study_manifest.json").exists()
assert (FIG_DIR / "study_summary_accuracy_efficiency.png").exists()
print(f"Execution checks passed: {expected_runs} runs and core artifacts are present.")

## 10. Conclusions

This notebook turns the supplied collection into one coherent, reproducible ODE study. It has:

- surveyed foundational PINNs, adaptive sampling, major PINN variants, DeepONet/FNO/PINO, and recent benchmarking/optimization developments;
- standardized nine ODE and system-ODE benchmarks, including exact or high-accuracy numerical references;
- reimplemented standard PINN, RAR, residual-adaptive sampling, DeepONet, and PI-DeepONet in a shared framework;
- generated quantitative accuracy, residual, time, iteration, memory-proxy, sampling, and convergence outputs automatically;
- separated forward-solver comparisons from an explicit data-informed inverse extension; and
- recorded constraints needed for responsible interpretation.

The best-performing approach in any execution is reported dynamically above rather than assumed in advance. The defensible high-level conclusion is methodological: adaptive sampling and operator learning answer different failure modes and should be selected by problem structure, data availability, anticipated number of parameter queries, and verification budget—not by a single aggregate rank.

## 11. References

The bibliography prioritizes peer-reviewed articles and conference papers. Entries marked *preprint* are included where they are historically influential or describe a recent development not yet verified here as a final archival version.

- **[L1]** Lagaris, I. E., Likas, A., & Fotiadis, D. I. (1998). Artificial neural networks for solving ordinary and partial differential equations. *IEEE Transactions on Neural Networks, 9*(5), 987–1000. [https://doi.org/10.1109/72.712178](https://doi.org/10.1109/72.712178)
- **[L2]** Raissi, M., Perdikaris, P., & Karniadakis, G. E. (2019). Physics-informed neural networks: A deep learning framework for solving forward and inverse problems involving nonlinear partial differential equations. *Journal of Computational Physics, 378*, 686–707. [https://doi.org/10.1016/j.jcp.2018.10.045](https://doi.org/10.1016/j.jcp.2018.10.045)
- **[L3]** Lu, L., Meng, X., Mao, Z., & Karniadakis, G. E. (2021). DeepXDE: A deep learning library for solving differential equations. *SIAM Review, 63*(1), 208–228. [https://doi.org/10.1137/19M1274067](https://doi.org/10.1137/19M1274067)
- **[L4]** Karniadakis, G. E., Kevrekidis, I. G., Lu, L., Perdikaris, P., Wang, S., & Yang, L. (2021). Physics-informed machine learning. *Nature Reviews Physics, 3*, 422–440. [https://doi.org/10.1038/s42254-021-00314-5](https://doi.org/10.1038/s42254-021-00314-5)
- **[L5]** Wang, S., Teng, Y., & Perdikaris, P. (2021). Understanding and mitigating gradient flow pathologies in physics-informed neural networks. *SIAM Journal on Scientific Computing, 43*(5), A3055–A3081. [https://doi.org/10.1137/20M1318043](https://doi.org/10.1137/20M1318043)
- **[L6]** Wang, S., Yu, X., & Perdikaris, P. (2022). When and why PINNs fail to train: A neural tangent kernel perspective. *Journal of Computational Physics, 449*, 110768. [https://doi.org/10.1016/j.jcp.2021.110768](https://doi.org/10.1016/j.jcp.2021.110768)
- **[L7]** McClenny, L., & Braga-Neto, U. (2023). Self-adaptive physics-informed neural networks using a soft attention mechanism. *Journal of Computational Physics, 474*, 111722. [https://doi.org/10.1016/j.jcp.2022.111722](https://doi.org/10.1016/j.jcp.2022.111722)
- **[L8]** Yu, J., Lu, L., Meng, X., & Karniadakis, G. E. (2022). Gradient-enhanced physics-informed neural networks for forward and inverse PDE problems. *Computer Methods in Applied Mechanics and Engineering, 393*, 114823. [https://doi.org/10.1016/j.cma.2022.114823](https://doi.org/10.1016/j.cma.2022.114823)
- **[L9]** Wu, C., Zhu, M., Tan, Q., Kartha, Y., & Lu, L. (2023). A comprehensive study of non-adaptive and residual-based adaptive sampling for physics-informed neural networks. *Computer Methods in Applied Mechanics and Engineering, 403*, 115671. [https://doi.org/10.1016/j.cma.2022.115671](https://doi.org/10.1016/j.cma.2022.115671)
- **[L10]** Nabian, M. A., Gladstone, R. J., & Meidani, H. (2021). Efficient training of physics-informed neural networks via importance sampling. *Computer-Aided Civil and Infrastructure Engineering, 36*(8), 962–977. [https://doi.org/10.1111/mice.12685](https://doi.org/10.1111/mice.12685)
- **[L11]** Gao, Z., Yan, L., & Zhou, T. (2023). Failure-informed adaptive sampling for PINNs. *SIAM Journal on Scientific Computing, 45*(4), A1971–A1994. [https://doi.org/10.1137/22M1527763](https://doi.org/10.1137/22M1527763)
- **[L12]** Jagtap, A. D., & Karniadakis, G. E. (2020). Extended physics-informed neural networks (XPINNs): A generalized space-time domain decomposition based deep learning framework for nonlinear partial differential equations. *Communications in Computational Physics, 28*(5), 2002–2041. [https://doi.org/10.4208/cicp.OA-2020-0164](https://doi.org/10.4208/cicp.OA-2020-0164)
- **[L13]** Jagtap, A. D., Kharazmi, E., & Karniadakis, G. E. (2020). Conservative physics-informed neural networks on discrete domains for conservation laws: Applications to forward and inverse problems. *Computer Methods in Applied Mechanics and Engineering, 365*, 113028. [https://doi.org/10.1016/j.cma.2020.113028](https://doi.org/10.1016/j.cma.2020.113028)
- **[L14]** Kharazmi, E., Zhang, Z., & Karniadakis, G. E. (2019). Variational physics-informed neural networks for solving partial differential equations. *arXiv preprint*. [https://arxiv.org/abs/1912.00873](https://arxiv.org/abs/1912.00873)
- **[L15]** Kharazmi, E., Zhang, Z., & Karniadakis, G. E. (2021). hp-VPINNs: Variational physics-informed neural networks with domain decomposition. *Computer Methods in Applied Mechanics and Engineering, 374*, 113547. [https://doi.org/10.1016/j.cma.2020.113547](https://doi.org/10.1016/j.cma.2020.113547)
- **[L16]** Moseley, B., Markham, A., & Nissen-Meyer, T. (2023). Finite basis physics-informed neural networks (FBPINNs): A scalable domain decomposition approach for solving differential equations. *Advances in Computational Mathematics, 49*, 62. [https://doi.org/10.1007/s10444-023-10065-9](https://doi.org/10.1007/s10444-023-10065-9)
- **[L17]** Liu, Z., Cai, W., & Xu, Z.-Q. J. (2020). Multi-scale deep neural network (MscaleDNN) for solving Poisson–Boltzmann equation in complex domains. *Communications in Computational Physics, 28*(5), 1970–2001. [https://doi.org/10.4208/cicp.OA-2020-0179](https://doi.org/10.4208/cicp.OA-2020-0179)
- **[L18]** Lu, L., Jin, P., Pang, G., Zhang, Z., & Karniadakis, G. E. (2021). Learning nonlinear operators via DeepONet based on the universal approximation theorem of operators. *Nature Machine Intelligence, 3*, 218–229. [https://doi.org/10.1038/s42256-021-00302-5](https://doi.org/10.1038/s42256-021-00302-5)
- **[L19]** Wang, S., Wang, H., & Perdikaris, P. (2021). Learning the solution operator of parametric partial differential equations with physics-informed DeepONets. *Science Advances, 7*, eabi8605. [https://doi.org/10.1126/sciadv.abi8605](https://doi.org/10.1126/sciadv.abi8605)
- **[L20]** Li, Z., Kovachki, N., Azizzadenesheli, K., et al. (2021). Fourier neural operator for parametric partial differential equations. *International Conference on Learning Representations*. [https://openreview.net/forum?id=c8P9NQVtmnO](https://openreview.net/forum?id=c8P9NQVtmnO)
- **[L21]** Li, Z., Zheng, H., Kovachki, N., et al. (2021). Physics-informed neural operator for learning partial differential equations. *arXiv preprint*. [https://arxiv.org/abs/2111.03794](https://arxiv.org/abs/2111.03794)
- **[L22]** Kovachki, N., et al. (2023). Neural operator: Learning maps between function spaces. *Journal of Machine Learning Research, 24*(89), 1–97. [https://www.jmlr.org/papers/v24/21-1524.html](https://www.jmlr.org/papers/v24/21-1524.html)
- **[L23]** Hao, Z., Yao, J., Su, C., et al. (2024). PINNacle: A comprehensive benchmark of physics-informed neural networks for solving PDEs. *Advances in Neural Information Processing Systems (Datasets and Benchmarks)*. [https://proceedings.neurips.cc/paper_files/paper/2024/hash/8c63299fb2820ef41cb05e2ff11836f5-Abstract-Datasets_and_Benchmarks_Track.html](https://proceedings.neurips.cc/paper_files/paper/2024/hash/8c63299fb2820ef41cb05e2ff11836f5-Abstract-Datasets_and_Benchmarks_Track.html)
- **[L24]** Wang, S., Sankaran, S., & Perdikaris, P. (2022). Respecting causality is all you need for training physics-informed neural networks. *arXiv preprint*. [https://arxiv.org/abs/2203.07404](https://arxiv.org/abs/2203.07404)
- **[L25]** Rathore, P., Lei, W., Frangella, Z., Lu, L., & Udell, M. (2024). Challenges in training PINNs: A loss landscape perspective. *International Conference on Machine Learning*. [https://arxiv.org/abs/2402.01868](https://arxiv.org/abs/2402.01868)
- **[L26]** Krishnapriyan, A., Gholami, A., Zhe, S., Kirby, R. M., & Mahoney, M. W. (2021). Characterizing possible failure modes in physics-informed neural networks. *Advances in Neural Information Processing Systems*. [https://arxiv.org/abs/2109.01050](https://arxiv.org/abs/2109.01050)
- **[L27]** Chen, R. T. Q., Rubanova, Y., Bettencourt, J., & Duvenaud, D. (2018). Neural ordinary differential equations. *Advances in Neural Information Processing Systems*. [https://arxiv.org/abs/1806.07366](https://arxiv.org/abs/1806.07366)

### Citation and search transparency

This is a focused survey, not a PRISMA-style systematic review. Claims about method definitions are linked to their primary or archival sources above. Recent items were checked against publisher/conference/author records available at the literature-snapshot date; readers should verify final bibliographic metadata before manuscript submission.

## 12. Reproducibility checklist

- [x] Single notebook, executable top-to-bottom after dependency bootstrap.
- [x] Deterministic seed and execution profile recorded.
- [x] Source-notebook provenance recorded without requiring source files at runtime.
- [x] Exact or high-accuracy numerical references isolated from PINN training.
- [x] Shared metrics: MSE, RMSE, relative L2, maximum absolute error, training time, iterations, residual, and memory proxy.
- [x] Figures: reference/prediction, error distributions, loss curves, residual evolution, sampling distributions, accuracy/time scatter, and heatmap.
- [x] CSV, JSON manifest, figures, and inverse-extension outputs automatically written under `pinn_ode_study_outputs/`.
- [ ] Multi-seed uncertainty intervals (recommended next step for a submission).
- [ ] Classical-solver timing baseline on the intended deployment hardware (recommended next step).